In [1]:
import polars as pl

In [2]:
df = pl.read_csv('data/dl_data.csv')
df.tail(5)

Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
str,str,i64,i64,f64,str,str,str,str,str,i64,str
"""10:57:01""","""2023-08-23""",2453933570,519744068,2247.25,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Small_Fan_Out"""
"""10:57:06""","""2023-08-23""",9805510177,5416607878,927.18,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Debit card""",0,"""Normal_Small_Fan_Out"""
"""10:57:06""","""2023-08-23""",7282330957,2995527149,1455.14,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Small_Fan_Out"""
"""10:57:11""","""2023-08-23""",940337377,4812815165,25995.7,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Fan_In"""
"""10:57:12""","""2023-08-23""",105185176,6824994831,9586.08,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Fan_Out"""


### NULL

In [3]:
df.null_count()

Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0


### DUPLICATE

In [4]:
df.filter(df.is_duplicated())

Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
str,str,i64,i64,f64,str,str,str,str,str,i64,str


### SUMMARY

In [5]:
for i in df.columns:
    print(f"{i}: {df[i].n_unique()} unique values")

Time: 86284 unique values
Date: 53 unique values
Sender_account: 103772 unique values
Receiver_account: 330291 unique values
Amount: 981755 unique values
Payment_currency: 13 unique values
Received_currency: 13 unique values
Sender_bank_location: 18 unique values
Receiver_bank_location: 18 unique values
Payment_type: 7 unique values
Is_laundering: 2 unique values
Laundering_type: 28 unique values


In [6]:
for i in df.columns:
    print('======================================')
    print(f"{i}: {df[i].n_unique()} unique values")
    print(df[i].unique())


Time: 86284 unique values
shape: (86_284,)
Series: 'Time' [str]
[
	"05:06:48"
	"07:10:35"
	"05:37:09"
	"18:55:17"
	"07:21:21"
	…
	"08:57:07"
	"05:36:22"
	"04:42:38"
	"01:15:52"
	"03:12:27"
]
Date: 53 unique values
shape: (53,)
Series: 'Date' [str]
[
	"2023-07-29"
	"2023-07-25"
	"2023-07-16"
	"2023-07-26"
	"2023-08-18"
	…
	"2023-08-05"
	"2023-08-14"
	"2023-07-28"
	"2023-07-15"
	"2023-07-12"
]
Sender_account: 103772 unique values
shape: (103_772,)
Series: 'Sender_account' [i64]
[
	9018
	28511
	92172
	155434
	162832
	…
	9999156867
	9999507228
	9999699792
	9999766666
	9999819588
]
Receiver_account: 330291 unique values
shape: (330_291,)
Series: 'Receiver_account' [i64]
[
	9018
	13266
	48238
	92172
	99278
	…
	9999899663
	9999914504
	9999933179
	9999967460
	9999971095
]
Amount: 981755 unique values
shape: (981_755,)
Series: 'Amount' [f64]
[
	4.86
	5.08
	6.78
	7.19
	7.41
	…
	3.1940e6
	3507884.1
	6.2132e6
	9.2164e6
	1.2359e7
]
Payment_currency: 13 unique values
shape: (13,)
Series: 'Payment_cu

### TIMESTAMP

In [7]:
df[['Time', 'Date']].schema

Schema([('Time', String), ('Date', String)])

### XỬ LÝ TIMESTAMP VÀ LƯU FILE

In [8]:
del df

In [10]:
(
    pl.scan_csv("data/dl_data.csv")
    .with_columns([
        pl.col("Time").str.strptime(pl.Time, "%H:%M:%S"),
        pl.col("Date").str.strptime(pl.Date, "%Y-%m-%d")
    ])
    .sink_csv("data/preprocessed_data.csv")
)